In [1]:
# Funciones auxiliares — Caso 1: Árbol de regresión (Car Price)
# Son las mismas que usamos en el laboratorio de árboles (caso MPG).
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   evaluar_regresion ahora también reporta el MAPE (%).
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)


def evaluar_regresion(nombre, y_true, y_pred):
    """Imprime y regresa R², RMSE, MAE y MAPE (%) de un modelo de regresión."""
    metricas = {
        'modelo': nombre,
        'R²': round(r2_score(y_true, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
        'MAPE (%)': round(mean_absolute_percentage_error(y_true, y_pred) * 100, 2),
    }
    print(f"{nombre}: R² {metricas['R²']} | RMSE {metricas['RMSE']} | "
          f"MAE {metricas['MAE']} | MAPE {metricas['MAPE (%)']}%")
    return metricas


def plot_importancias(modelo, columnas, top_n=None):
    """Importancia de cada variable en un modelo basado en árboles."""
    imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values(ascending=True)
    if top_n:
        imp = imp.tail(top_n)

    fig = px.bar(
        x=imp.values, y=imp.index, orientation='h', text_auto='.3f',
        labels={'x': 'Importancia (reducción de impureza)', 'y': ''},
        title='Importancia de variables', template='plotly_white'
    )
    fig.update_layout(width=800, height=450)
    fig.show()
    return imp.sort_values(ascending=False)


#Caso 2

# Funciones auxiliares — Caso 2: Regresión logística (Phone Classification)
# Son las mismas que usamos en los laboratorios de regresión logística y árboles.
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   evaluar_clasificador y plot_matriz_confusion se ajustaron para funcionar
#      con MÁS DE DOS clases (en Titanic sólo había dos: sobrevivió / no sobrevivió).
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.preprocessing import StandardScaler


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def plot_rfecv_clasificacion(rfecv, metrica='exactitud'):
    """Desempeño promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name=f'{metrica} promedio (CV)'
    ))
    fig.add_vline(x=rfecv.n_features_, line_dash='dash', line_color='red',
                  annotation_text=f'elegido: {rfecv.n_features_}')
    fig.update_layout(
        title='RFECV — desempeño según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title=f'{metrica.capitalize()} (validación cruzada)',
        template='plotly_white', width=900, height=450
    )
    fig.show()


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud, precisión, recall y F1.

    Con más de dos clases, precisión, recall y F1 son el promedio simple
    de las clases (average='macro').
    """
    binario = len(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)]))) <= 2
    promedio = 'binary' if binario else 'macro'
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 4),
        'precisión': round(precision_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, average=promedio, zero_division=0), 4),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_matriz_confusion(y_true, y_pred, clases=None, etiquetas=None, title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos.

    clases: los valores de la variable objetivo en el orden en que quieres verlos,
            p. ej. ['Low', 'Mid_Low', 'Mid', 'High']. Si no las pasas, se ordenan alfabéticamente.
    etiquetas: nombres a mostrar para esas clases (opcional; por defecto, las mismas clases).
    """
    if clases is None:
        clases = sorted(set(np.asarray(y_true)) | set(np.asarray(y_pred)))
    clases = list(clases)
    etiquetas = list(etiquetas) if etiquetas is not None else clases

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=clases),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()
    return cm


CASO 2 


In [2]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/CarPrice_dataset.csv')

# Solo columnas numéricas
num = df.select_dtypes(include='number')
X = num.drop(columns=['price'])
y = num['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

arbol = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

imp = pd.Series(arbol.feature_importances_, index=X.columns).sort_values()
px.bar(x=imp.values, y=imp.index, orientation='h',
       title='Importancia de variables', labels={'x': 'Importancia', 'y': ''}).show()

print(imp.sort_values(ascending=False))
print("Más importante:", imp.idxmax())

enginesize          0.675420
curbweight          0.235578
carwidth            0.019698
compressionratio    0.016401
peakrpm             0.015779
horsepower          0.011656
stroke              0.006883
carlength           0.005857
symboling           0.004067
citympg             0.003412
wheelbase           0.002212
highwaympg          0.001147
boreratio           0.000769
carheight           0.000670
doornumber          0.000377
cylindernumber      0.000073
dtype: float64
Más importante: enginesize


LA MAS IMPORTANTE ES ENGINESIZE CON UN VALOR DE 0.6754 

In [3]:
from sklearn.metrics import r2_score

X2 = num[['enginesize', 'curbweight']]
y = num['price']

X_train, X_test, y_train, y_test = train_test_split(X2, y, test_size=0.2, random_state=0)

modelo = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("R2 test:", r2_score(y_test, modelo.predict(X_test)))

print("X_train.shape =", X_train.shape)
print("X_test.shape =", X_test.shape)
print("y_train.shape =", y_train.shape)
print("y_test.shape =", y_test.shape)

R2 test: 0.8885901518557542
X_train.shape = (164, 2)
X_test.shape = (41, 2)
y_train.shape = (164,)
y_test.shape = (41,)


In [4]:
r2_train = modelo.score(X_train, y_train)
r2_test = modelo.score(X_test, y_test)

print(f"R2_train = {r2_train*100:.2f}")
print(f"R2_test = {r2_test*100:.2f}")
print(f"Diferencia = {(r2_train - r2_test)*100:.2f}")

R2_train = 99.83
R2_test = 88.86
Diferencia = 10.97


In [5]:
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

pred_train = modelo.predict(X_train)
pred_test = modelo.predict(X_test)

print(f"MAPE_train = {mean_absolute_percentage_error(y_train, pred_train)*100:.2f}")
print(f"MAPE_test = {mean_absolute_percentage_error(y_test, pred_test)*100:.2f}")
print(f"RMSE_train = {np.sqrt(mean_squared_error(y_train, pred_train)):.2f}")
print(f"RMSE_test = {np.sqrt(mean_squared_error(y_test, pred_test)):.2f}")

MAPE_train = 0.77
MAPE_test = 18.51
RMSE_train = 322.88
RMSE_test = 2936.83


Caso 1

In [6]:
import pandas as pd
import plotly.express as px

url = "https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/phone_classification.csv"
df = pd.read_csv(url)

conteo = df['price_range'].value_counts().reset_index()
conteo.columns = ['price_range', 'cantidad']

fig = px.bar(conteo, x='price_range', y='cantidad', text='cantidad',
             title='Cantidad de observaciones por price_range',
             labels={'price_range': 'Rango de precio', 'cantidad': 'Número de observaciones'})
fig.show()

In [7]:
orden = {'Low': 0, 'Mid_Low': 1, 'Mid': 2, 'High': 3}
df['price_num'] = df['price_range'].map(orden)

corr = (df.drop(columns=['price_range'])
          .corr(numeric_only=True)['price_num']
          .drop('price_num'))

print(corr.sort_values(ascending=False))
print("Mayor relación:", corr.abs().idxmax())
print("Menor relación:", corr.abs().idxmin())

ram              0.921647
battery_power    0.206876
px_width         0.171549
px_height        0.156043
int_memory       0.047408
blue             0.031410
fc               0.029378
talk_time        0.025332
wifi             0.021363
dual_sim         0.016497
four_g           0.015491
m_dep            0.004895
n_cores          0.002594
clock_speed     -0.002126
touch_screen    -0.029865
mobile_wt       -0.030850
Name: price_num, dtype: float64
Mayor relación: ram
Menor relación: clock_speed


variable con mayor relacion es "ram" y con menor es "clock_speed"


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV

X = df.drop(columns=['price_range', 'price_num'])
y = df['price_range']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

escalador = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X.columns, index=X_test.index)

logit = LogisticRegression(random_state=0, max_iter=500)
rfecv = RFECV(estimator=logit, cv=5).fit(X_train_s, y_train)

print("Número óptimo de variables:", rfecv.n_features_)
print("Variables seleccionadas:", list(X.columns[rfecv.support_]))

Número óptimo de variables: 5
Variables seleccionadas: ['battery_power', 'mobile_wt', 'px_height', 'px_width', 'ram']


Las variables son 5 y ya me da los nombres de las 5 elegidas

In [9]:
sel = ['battery_power', 'mobile_wt', 'px_height', 'px_width', 'ram']
corr_sel = X_train[sel].corr().round(2)

px.imshow(corr_sel, text_auto=True, color_continuous_scale='RdBu_r',
          zmin=-1, zmax=1, title='Correlación entre variables seleccionadas').show()

# Mayor correlación entre pares distintos (sin la diagonal)
import numpy as np
m = corr_sel.abs().where(~np.eye(len(sel), dtype=bool))
print("Máxima |correlación|:", m.max().max())

Máxima |correlación|: 0.49


la multicolinealidad no hay ya que la max correlacion da 0.49 y debe superar el 0.80 

In [10]:
from sklearn.metrics import accuracy_score

sel = ['battery_power', 'mobile_wt', 'px_height', 'px_width', 'ram']
Xtr = X_train_s[sel]
Xte = X_test_s[sel]

logit_sel = LogisticRegression(random_state=0, max_iter=500).fit(Xtr, y_train)

pred_train = logit_sel.predict(Xtr)
pred_test = logit_sel.predict(Xte)

acc_train = accuracy_score(y_train, pred_train)
acc_test = accuracy_score(y_test, pred_test)

print(f"Accuracy_train = {acc_train:.4f}")
print(f"Accuracy_test = {acc_test:.4f}")
print(f"Diferencia = {acc_train - acc_test:.4f}")

Accuracy_train = 0.9723
Accuracy_test = 0.9676
Diferencia = 0.0047


la diferencia para overfitting debe superar 10, no llega ni a 1 

In [11]:
from sklearn.metrics import confusion_matrix

etiquetas = ['Low', 'Mid_Low', 'Mid', 'High']
cm = confusion_matrix(y_test, pred_test, labels=etiquetas)

cm_df = pd.DataFrame(cm,
                     index=[f'Real: {e}' for e in etiquetas],
                     columns=[f'Predicho: {e}' for e in etiquetas])

fig = px.imshow(cm_df, text_auto=True, color_continuous_scale='Blues',
                title='Matriz de confusión - Regresión Logística (test)')
fig.show()